# QuasiShuffleAlgebra.jl — Complete Guide

A Julia package implementing the quasi-shuffle algebra of MacMahonesque partition functions
for partition-theoretic prime detection, based on:

> **"Integer Partitions Detect the Primes"**  
> William Craig, Jan-Willem van Ittersum & Ken Ono  
> arXiv:2405.06451v2, 2024

**Key result (Theorem 1.1):** There exist explicit non-negative expressions $E_i(n)$
built from MacMahon partition functions such that for $n \geq 2$:

$$E_i(n) = 0 \iff n \text{ is prime}$$

This notebook demonstrates every component of the package.

## 1. Installation and Setup

In [ ]:
# Activate the QuasiShuffleAlgebra project
using Pkg
Pkg.activate(joinpath(@__DIR__, "QuasiShuffleAlgebra"))
Pkg.instantiate()

using QuasiShuffleAlgebra
using Printf

## 2. Type System: `Word` and `ZqElem`

The package uses two core type aliases:

- **`Word = Vector{Int}`** — A finite sequence of non-negative integers (a "word" or exponent vector)
- **`ZqElem = Dict{Word, Rational{BigInt}}`** — A formal linear combination of words with exact rational coefficients

A `ZqElem` represents an element of the quasi-shuffle algebra $\mathcal{Z}_q$.

In [ ]:
# Creating words
w1 = Word([1])        # single letter [1]
w2 = Word([1, 1])     # two-letter word [1,1]
w3 = Word([2, 1, 3])  # three-letter word [2,1,3]

println("Words: ", w1, ", ", w2, ", ", w3)

In [ ]:
# Creating ZqElem objects (elements of the quasi-shuffle algebra)
elem = ZqElem(
    [1, 1] => Rational{BigInt}(3),
    [3]    => Rational{BigInt}(1, 6),
    [1]    => Rational{BigInt}(-1, 6)
)

println("A ZqElem:")
for (word, coeff) in sort(collect(elem), by = p -> (length(p.first), p.first))
    println("  ", coeff, " * M_", word)
end

In [ ]:
# Arithmetic helpers: zq_add, zq_scale, zq_sub, cleanup!
a = ZqElem([1] => Rational{BigInt}(2), [2] => Rational{BigInt}(3))
b = ZqElem([1] => Rational{BigInt}(-1), [3] => Rational{BigInt}(5))

c = zq_add(a, b)
println("a + b = ", c)

d = zq_scale(Rational{BigInt}(1, 2), a)
println("(1/2) * a = ", d)

e = zq_sub(a, b)
println("a - b = ", e)

In [ ]:
# evaluate_zq: evaluate a ZqElem at a specific n
# This computes sum_w c_w * M_w(n) using M_macmahonesque
elem = ZqElem([1] => Rational{BigInt}(1))  # just M_{[1]}(n) = sigma_1(n)

for n in 1:10
    val = evaluate_zq(elem, n)
    @printf "  n=%2d  evaluate_zq = %s  (sigma_1 = %d)\n" n val σ(1, n)
end

## 3. Divisor Power Sums and MacMahon Functions

### Divisor power sum $\sigma_k(n) = \sum_{d \mid n} d^k$

### MacMahon functions (closed forms from Fourier expansions):
- $M_1(n) = \sigma_1(n)$
- $M_2(n) = \frac{1}{8}[(-2n+1)\sigma_1(n) + \sigma_3(n)]$
- $M_3(n) = \frac{1}{1920}[(40n^2-100n+37)\sigma_1(n) - 10(3n-5)\sigma_3(n) + 3\sigma_5(n)]$

In [ ]:
# Divisor power sums
println("Divisor power sums sigma_k(n):")
println("─" ^ 50)
for n in 1:12
    @printf "  n=%2d  σ₀=%2d  σ₁=%3d  σ₂=%4d  σ₃=%5d\n" n σ(0,n) σ(1,n) σ(2,n) σ(3,n)
end

In [ ]:
# MacMahon functions M1, M2, M3 (closed-form)
println("MacMahon function values:")
println("─" ^ 55)
@printf "  %3s  %6s  %10s  %15s\n" "n" "M₁(n)" "M₂(n)" "M₃(n)"
println("  ", "─" ^ 50)
for n in 1:15
    @printf "  %3d  %6d  %10s  %15s\n" n M1(n) M2(n) M3(n)
end

## 4. MacMahonesque Functions

The **MacMahonesque function** $M_{\vec{a}}(n)$ generalizes MacMahon functions with a
weight vector $\vec{a} = (v_1, v_2, \ldots, v_a)$:

$$M_{\vec{a}}(n) = \sum_{\substack{0 < s_1 < s_2 < \cdots < s_a \\ m_i \geq 1 \\ \sum m_i s_i = n}} m_1^{v_1} m_2^{v_2} \cdots m_a^{v_a}$$

Special cases:
- $M_{(1)}(n) = \sigma_1(n)$ (sum of divisors)
- $M_{(k)}(n) = \sigma_k(n)$ (k-th power sum of divisors)
- $M_{(1,1,\ldots,1)}(n) = M_a(n)$ (classical MacMahon)

### ⚠️ Weight Convention

**`vec_a[1]` is the exponent of the *largest* part**, not the smallest.

Definition (1.3) of the paper writes $m_1^{v_1} \cdots m_a^{v_a}$ with $s_1 < s_2 < \cdots < s_a$,
suggesting $v_1$ is the exponent of the smallest part. But **every explicit formula in the paper**
(Table 1, the D-operator example on p. 4, etc.) uses the opposite convention:
$v_1$ applies to the **largest** part.

This was discovered by checking the D-operator identity $n \cdot M_{(1,1)}(n)$ at $n=4$:
- Under the "definition" convention: LHS = 12, RHS = 180/11 ✗
- Under the "formula" convention (`vec_a[1]` = largest part): LHS = 12, RHS = 12 ✓

The implementation uses the **formula convention** throughout.

In [ ]:
# Demonstrate the weight convention at n=4
# n·M_{(1,1)}(4) should equal 4 * M_macmahonesque([1,1], 4) = 4 * 3 = 12
n = 4
lhs = n * M_macmahonesque([1, 1], n, Rational{BigInt})
println("n·M_{(1,1)}(4) = $lhs")   # should be 12

# The paper's p.4 formula (explicit D-operator result) evaluated at n=4:
# (1/22)[-21·M_{(3,1)} + 72·M_{(2,2)} - 9·M_{(1,3)} + 24·M_{(3,0)}
#         - 24·M_{(3,0,0)} - 24·M_{(2,1,0)} + 24·M_{(2,0,1)} - 72·M_{(1,1,1)}]
# vec_a[1] = exponent of LARGEST part
paper_rhs = (
    -21 * M_macmahonesque([3, 1], n, Rational{BigInt}) +
     72 * M_macmahonesque([2, 2], n, Rational{BigInt}) +
     -9 * M_macmahonesque([1, 3], n, Rational{BigInt}) +
     24 * M_macmahonesque([3, 0], n, Rational{BigInt}) +
    -24 * M_macmahonesque([3, 0, 0], n, Rational{BigInt}) +
    -24 * M_macmahonesque([2, 1, 0], n, Rational{BigInt}) +
     24 * M_macmahonesque([2, 0, 1], n, Rational{BigInt}) +
    -72 * M_macmahonesque([1, 1, 1], n, Rational{BigInt})
) // 22
println("Paper formula at n=4 = $paper_rhs")   # should also be 12
println("Match: $(lhs == paper_rhs)")

In [ ]:
# M_macmahonesque with various weight vectors
println("MacMahonesque function values for selected weight vectors:")
println("─" ^ 65)

vecs = [[1], [2], [3], [1,1], [2,1], [1,1,1]]
labels = ["(1)", "(2)", "(3)", "(1,1)", "(2,1)", "(1,1,1)"]

@printf "  %3s" "n"
for l in labels
    @printf "  %10s" "M_$l"
end
println()
println("  ", "─" ^ 62)

for n in 1:12
    @printf "  %3d" n
    for v in vecs
        @printf "  %10d" M_macmahonesque(v, n)
    end
    println()
end

In [ ]:
# Single-letter MacMahonesque = divisor power sum
println("Verify: M_{(k)}(n) = σ_k(n)")
for n in 1:10
    for k in 1:4
        @assert M_macmahonesque([k], n) == σ(k, n)
    end
end
println("  All checks passed for n=1:10, k=1:4")

## 5. `M_direct` vs `M_macmahonesque` Consistency

`M_direct(a, n)` computes $M_a(n)$ (the classical MacMahon function with all-ones weight vector)
via direct combinatorial enumeration. It should agree with `M_macmahonesque(ones(a), n)`.

In [ ]:
# Cross-check M_direct(a, n) vs M_macmahonesque(ones(a), n)
println("Consistency check: M_direct(a, n) == M_macmahonesque(ones(a), n)")
println("─" ^ 55)

all_ok = true
for a in 1:4
    ones_a = ones(Int, a)
    for n in 1:30
        v1 = M_direct(a, n)
        v2 = M_macmahonesque(ones_a, n)
        if v1 != v2
            println("  MISMATCH at a=$a, n=$n: M_direct=$v1, M_macmahonesque=$v2")
            all_ok = false
        end
    end
end

if all_ok
    println("  All checks passed for a=1:4, n=1:30")
end

In [ ]:
# Also verify closed-form M1, M2, M3 against M_direct
println("Closed-form vs combinatorial:")
for n in 1:20
    @assert M1(n) == M_direct(1, n) "M1 mismatch at n=$n"
    @assert M2(n) == M_direct(2, n) "M2 mismatch at n=$n"
    @assert M3(n) == M_direct(3, n) "M3 mismatch at n=$n"
end
println("  M1, M2, M3 all match M_direct for n=1:20")

## 6. Prime Detection: Expressions $E_1$ through $E_4$

The paper proves that certain polynomial combinations of MacMahon functions detect primes:

$$E_i(n) \geq 0 \text{ for all } n \geq 1, \qquad E_i(n) = 0 \iff n \text{ is prime} \; (n \geq 2)$$

| Expression | Formula | Quasimodular form |
|---|---|---|
| $E_1$ | $(n^2-3n+2)M_1(n) - 8M_2(n)$ | $6H_6$ |
| $E_2$ | $(3n^3-13n^2+18n-8)M_1 + (12n^2-120n+212)M_2 - 960M_3$ | $36H_8$ |
| $E_3$ | Uses $M_1$ through $M_4$ | $90H_{10}$ |
| $E_4$ | Uses $M_1$ through $M_5$ | $90H_{12}$ |

In [ ]:
# E1: the simplest prime-detecting expression
println("E1(n) = (n²-3n+2)·M₁(n) - 8·M₂(n)")
println("─" ^ 45)
println("  (E1(n) = 0 iff n is prime)")
println()

for n in 2:25
    val = E1(n)
    marker = iszero(val) ? "  <-- PRIME" : ""
    @printf "  n=%2d  E1 = %s%s\n" n val marker
end

In [ ]:
# E2: higher-order prime detection
println("E2(n) — zero at primes, positive at composites:")
println("─" ^ 50)

for n in 2:20
    val = E2(n)
    marker = iszero(val) ? "  <-- PRIME" : ""
    @printf "  n=%2d  E2 = %s%s\n" n val marker
end

In [ ]:
# Compare all four expressions at selected values
println("Comparison of E1-E4 at selected n:")
println("─" ^ 70)
@printf "  %3s  %8s  %12s  %15s  %18s  %s\n" "n" "E1" "E2" "E3" "E4" "Prime?"
println("  ", "─" ^ 67)

for n in [2, 3, 4, 5, 6, 7, 10, 11, 12, 13]
    e1 = E1(n)
    e2 = E2(n)
    e3 = E3(n)
    e4 = E4(n)
    is_p = all(iszero, [e1, e2, e3, e4]) ? "yes" : "no"
    @printf "  %3d  %8s  %12s  %15s  %18s  %s\n" n e1 e2 e3 e4 is_p
end

## 7. Partition-Theoretic Primality Test

`is_prime_partition(n)` returns `true` iff $E_1(n) = 0$, which is equivalent to $n$ being prime.

`verify_range(lo, hi)` cross-checks against trial division over an entire range.

In [ ]:
# Find primes in [2, 50] using partition theory
primes_partition = filter(is_prime_partition, 2:50)
println("Primes in [2, 50] via partition test:")
println("  ", primes_partition)

In [ ]:
# Verify against trial division over [2, 200]
println("Verification against trial division:")
verify_range(2, 200)

## 8. Bernoulli Numbers

The diamond product uses **Bernoulli numbers** $B_n$, computed exactly via the recurrence:

$$B_n = -\frac{1}{n+1} \sum_{k=0}^{n-1} \binom{n+1}{k} B_k$$

Key values: $B_0 = 1$, $B_1 = -1/2$, $B_{2k+1} = 0$ for $k \geq 1$, $B_{12} = -691/2730$.

In [ ]:
# Bernoulli numbers B_0 through B_20
println("Bernoulli numbers (exact rational):")
println("─" ^ 40)

for n in 0:20
    b = bernoulli(n)
    if !iszero(b)
        @printf "  B_%2d = %s\n" n b
    end
end

In [ ]:
# Verify key values
@assert bernoulli(0)  == 1
@assert bernoulli(1)  == -1//2
@assert bernoulli(12) == -691//2730
@assert all(iszero(bernoulli(n)) for n in 3:2:19)  # odd Bernoulli = 0

println("All Bernoulli number checks passed.")

## 9. Diamond Product

The **diamond product** $i \diamond j$ combines two letters into a linear combination of letters.
It is defined by equation (4.3) in the paper:

$$i \diamond j = \frac{i! \, j!}{(i+j+1)!} \cdot [i+j+1] + \sum_{m=0}^{i+j} \frac{[(-1)^i \binom{i}{m} + (-1)^j \binom{j}{m}] \cdot B_{i+j+1-m}}{i+j+1-m} \cdot [m]$$

This product encodes how two generating series combine multiplicatively.

In [ ]:
# Diamond product examples
println("Diamond product coefficients:")
println("─" ^ 50)

for (i, j) in [(1,1), (1,2), (2,1), (2,2), (1,3)]
    coeffs = diamond_coeffs(i, j)
    print("  $i ◇ $j = ")
    terms = sort(collect(coeffs), by = p -> p.first)
    parts = String[]
    for (m, c) in terms
        push!(parts, "$(c)·[$m]")
    end
    println(join(parts, " + "))
end

In [ ]:
# The diamond function returns a full ZqElem (single-letter words)
d11 = diamond(1, 1)
println("diamond(1,1) as ZqElem:")
for (word, coeff) in sort(collect(d11), by = p -> p.first)
    println("  ", coeff, " * M_", word)
end

## 10. Quasi-Shuffle Product

The **quasi-shuffle product** $*$ is the core algebraic operation. For words $xw$ and $yv$:

$$xw * yv = x(w * yv) + y(xw * v) + (x \diamond y)(w * v)$$

with base cases $\varepsilon * w = w * \varepsilon = w$ (empty word is identity).

**Ground truth** from the paper:
$$[1] * [1,1] = 3 \cdot [1,1,1] + \tfrac{1}{6}[3,1] + \tfrac{1}{6}[1,3] - \tfrac{1}{3}[1,1]$$

In [ ]:
# Ground truth: quasishuffle([1], [1,1])
result = quasishuffle_words([1], [1, 1])

println("[1] * [1,1] =")
for (word, coeff) in sort(collect(result), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

# Verify against known answer
expected = ZqElem(
    [1, 1, 1] => Rational{BigInt}(3),
    [3, 1]    => Rational{BigInt}(1, 6),
    [1, 3]    => Rational{BigInt}(1, 6),
    [1, 1]    => Rational{BigInt}(-1, 3)
)

# Check each coefficient
all_match = true
for (w, c) in expected
    if get(result, w, Rational{BigInt}(0)) != c
        println("  MISMATCH at $w: got $(get(result, w, 0)), expected $c")
        all_match = false
    end
end
println(all_match ? "\nGround truth verified!" : "\nMismatch detected!")

In [ ]:
# More quasi-shuffle examples
println("[1] * [1]:")
r11 = quasishuffle_words([1], [1])
for (word, coeff) in sort(collect(r11), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

println("\n[2] * [1]:")
r21 = quasishuffle_words([2], [1])
for (word, coeff) in sort(collect(r21), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

## 11. Convolution Identity Verification

The quasi-shuffle product encodes **convolution of partition functions**:

$$\sum_{i+j=n} M_{\vec{a}}(i) \cdot M_{\vec{b}}(j) = \sum_{\vec{w}} c_{\vec{w}} \cdot M_{\vec{w}}(n)$$

where the RHS coefficients $c_{\vec{w}}$ come from $[\vec{a}] * [\vec{b}]$.

We verify this numerically for $[1] * [1]$.

In [ ]:
# Convolution identity for [1] * [1]
qs_result = quasishuffle_words([1], [1])

println("Convolution identity: sum_{i+j=n} M_(1)(i) * M_(1)(j) = ...")
println("─" ^ 55)

all_ok = true
for n in 1:30
    # LHS: convolution
    lhs = sum(Rational{BigInt}(M_macmahonesque([1], i, Rational{BigInt})) *
              Rational{BigInt}(M_macmahonesque([1], n - i, Rational{BigInt}))
              for i in 0:n;  init=Rational{BigInt}(0))

    # RHS: evaluate quasi-shuffle result
    rhs = evaluate_zq(qs_result, n)

    match = lhs == rhs
    if !match
        println("  n=$n: MISMATCH lhs=$lhs rhs=$rhs")
        all_ok = false
    end
end

println(all_ok ? "  All checks passed for n=1:30" : "  Some mismatches found!")

## 12. `zq_multiply` — Multiplying ZqElem Objects

`zq_multiply(a, b)` extends the quasi-shuffle product from words to full `ZqElem` objects
by distributing linearly over all word pairs.

In [ ]:
# Multiply two ZqElems
a = ZqElem([1] => Rational{BigInt}(1))   # M_{(1)}
b = ZqElem([1] => Rational{BigInt}(1))   # M_{(1)}

prod_ab = zq_multiply(a, b)

println("M_{(1)} * M_{(1)} =")
for (word, coeff) in sort(collect(prod_ab), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

In [ ]:
# Multiply more complex elements
c = ZqElem([1] => Rational{BigInt}(2), [2] => Rational{BigInt}(-1))  # 2*M_(1) - M_(2)
d = ZqElem([1] => Rational{BigInt}(1))                                # M_(1)

prod_cd = zq_multiply(c, d)

println("(2·M_(1) - M_(2)) * M_(1) =")
for (word, coeff) in sort(collect(prod_cd), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

## 13. The D Operator

The **D operator** expresses $n \cdot M_{\vec{a}}(n)$ as a constant-coefficient linear
combination of MacMahonesque functions:

$$n \cdot M_{\vec{a}}(n) = \sum_{\vec{w}} c_{\vec{w}} \cdot M_{\vec{w}}(n)$$

This is computed by solving an exact linear system over $\mathbb{Q}$ via Gaussian elimination
with `Rational{BigInt}` arithmetic.

In [ ]:
# D operator applied to [1]: express n*M_(1)(n) as MacMahonesque combination
println("Computing d_operator([1])...")
d1 = d_operator([1])

println("n · M_(1)(n) =")
for (word, coeff) in sort(collect(d1), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

In [ ]:
# Verify numerically: n*M_(1)(n) should equal the linear combination
println("Numerical verification of d_operator([1]):")
println("─" ^ 45)

all_ok = true
for n in 1:20
    lhs = Rational{BigInt}(n) * M_macmahonesque([1], n, Rational{BigInt})
    rhs = evaluate_zq(d1, n)
    match = lhs == rhs
    @printf "  n=%2d  n·M_(1) = %6s  D-expansion = %6s  %s\n" n lhs rhs (match ? "ok" : "FAIL")
    all_ok &= match
end
println(all_ok ? "\nAll checks passed!" : "\nSome mismatches!")

In [ ]:
# D operator for [1,1] (more complex)
println("Computing d_operator([1,1])... (this may take a moment)")
d11 = d_operator([1, 1])

println("n · M_(1,1)(n) =")
for (word, coeff) in sort(collect(d11), by = p -> (length(p.first), p.first))
    println("  ", coeff, " · M_", word)
end

# Verify
println("\nNumerical verification (n=1:20):")
all_ok = all(1:20) do n
    lhs = Rational{BigInt}(n) * M_macmahonesque([1, 1], n, Rational{BigInt})
    rhs = evaluate_zq(d11, n)
    lhs == rhs
end
println(all_ok ? "  All checks passed!" : "  Mismatches found!")

### Paper's Explicit Formula (p. 4)

The paper gives the decomposition:

$$n \cdot M_{(1,1)}(n) = \frac{1}{22}\bigl[
  -21 M_{(3,1)} + 72 M_{(2,2)} - 9 M_{(1,3)} + 24 M_{(3,0)}
  - 24 M_{(3,0,0)} - 24 M_{(2,1,0)} + 24 M_{(2,0,1)} - 72 M_{(1,1,1)}
\bigr](n)$$

This explicit formula verifies the D operator's RREF-based solver and confirms the **weight convention**
(`vec_a[1]` = exponent of largest part).

Note: `d_operator` may return a *different* decomposition — MacMahonesque basis functions
can be linearly dependent as integer sequences, so the particular solution from RREF
is one of many equivalent representations. Both produce identical numerical values.

In [ ]:
# Paper's explicit D-operator formula for n*M_{(1,1)}(n)
# (from p.4 of Craig-van Ittersum-Ono 2024)
paper_d11 = ZqElem(
    [3, 1]    => -21//big(22),
    [2, 2]    =>  72//big(22),
    [1, 3]    =>  -9//big(22),
    [3, 0]    =>  24//big(22),
    [3, 0, 0] => -24//big(22),
    [2, 1, 0] => -24//big(22),
    [2, 0, 1] =>  24//big(22),
    [1, 1, 1] => -72//big(22),
)

println("Paper formula (n·M_{(1,1)})  vs  d_operator result:")
println("─" ^ 60)
all_ok = true
for n in 1:20
    lhs   = Rational{BigInt}(n) * M_macmahonesque([1, 1], n, Rational{BigInt})
    paper = evaluate_zq(paper_d11, n)
    dop   = evaluate_zq(d_operator([1, 1]), n)   # use cached value from above
    match_paper = lhs == paper
    match_dop   = lhs == dop
    all_ok &= match_paper & match_dop
    @printf "  n=%2d  n·M₂=%8s  paper=%8s  d_op=%8s  [%s]\n" n lhs paper dop (match_paper && match_dop ? "ok" : "FAIL")
end
println(all_ok ? "\nAll three agree for n=1:20!" : "\nMismatches found.")

## 14. Symmetrisation

The **symmetrisation** of a word $\vec{a}$ sums over all unique permutations:

$$\text{Sym}(\vec{a}) = \sum_{\pi \in S_{\text{unique}}} M_{\pi(\vec{a})}$$

When all entries of $\vec{a}$ are odd, the symmetrised series is a **quasimodular form**.

In [ ]:
# Symmetrise various words
examples = [[1, 1], [1, 2], [1, 1, 1], [2, 1], [1, 3]]

for vec_a in examples
    sym = symmetrise(vec_a)
    println("Sym(", vec_a, ") =")
    for (word, coeff) in sort(collect(sym), by = p -> (length(p.first), p.first))
        println("  ", coeff, " · M_", word)
    end
    println()
end

In [ ]:
# Symmetrisation of [1,1] should give 2*M_{(1,1)}
# (since [1,1] has only one unique permutation: itself)
sym11 = symmetrise([1, 1])
println("Sym([1,1]):")
for (word, coeff) in sym11
    println("  ", coeff, " · M_", word)
end

# Symmetrisation of [1,2]: permutations are [1,2] and [2,1]
sym12 = symmetrise([1, 2])
println("\nSym([1,2]):")
for (word, coeff) in sort(collect(sym12), by = p -> p.first)
    println("  ", coeff, " · M_", word)
end

## 15. Word Enumeration

`all_words_up_to_weight(w; max_length, min_length)` generates all words with non-negative
integer entries summing to at most `w`, within the given length bounds. This is used
internally by the D operator to construct its basis.

In [ ]:
# Enumerate words up to weight 3, length 1 to 2
words = all_words_up_to_weight(3; max_length=2, min_length=1)

println("Words with weight <= 3, length in [1,2]:")
for w in sort(words, by = w -> (length(w), w))
    println("  ", w, "  (weight=$(sum(w)), length=$(length(w)))")
end
println("Total: ", length(words), " words")

In [ ]:
# Larger enumeration — shows growth rate
for w in 1:6
    count = length(all_words_up_to_weight(w; max_length=3, min_length=1))
    println("  Weight <= $w, length <= 3: $count words")
end

## 16. Open Conjecture: Computational Test

The paper poses an open conjecture:

> **Conjecture.** Any expression $E(n) = \sum_i p_i(n) \cdot M_i(n) \geq 0$ that vanishes
> precisely at primes is a $\mathbb{Q}[n]$-linear combination of the five entries in Table 1.

The package provides tools to test this computationally over finite polynomial degree and weight bounds.

### Strategy

Fix degree bound $d$ and weight bound $a_\text{max}$:

1. **Build basis** $B = \{n^k \cdot M_a(n) : 0 \leq k \leq d,\; 1 \leq a \leq a_\text{max}\}$
2. **Prime-vanishing subspace**: null space of the prime evaluation matrix over $\mathbb{Q}$
3. **Table 1 $\mathbb{Q}[n]$-span**: for each Table 1 entry $E_i$ and $j = 0,\ldots,d$, include $n^j \cdot E_i(n)$ as a generator
4. **Check**: does every null-space vector lie in this span?

This tests the *linear* part of the conjecture (vanishing at primes); non-negativity at composites is a separate positivity question.

### Key result

| $(d, a_\text{max})$ | Basis size | pv-dim | t1-rank | Holds? |
|---|---|---|---|---|
| (2, 2) | 6 | 1 | 12 | ✓ (E1 alone suffices) |
| (3, 4) | 16 | 6 | 16 | ✓ (E1–E4 span) |
| (4, 4) | 20 | 9 | 20 | ✓ |
| (3, 5) | 20 | 8 | 16 | E5 needed |

At $a_\text{max} = 5$, $M_5$ generates new prime-vanishing directions that require $E_5$ (the
unknown fifth Table 1 entry) to close the span.

In [ ]:
# Build a basis and inspect its structure
d, a_max = 2, 3
basis = build_basis(d, a_max)
println("Basis for (d=$d, a_max=$a_max): $(length(basis)) elements")
println("  Format: (k, a) means n^k · M_a(n)")
for (k, a) in basis
    println("  n^$k · M_$a")
end

In [ ]:
# Rational null space: basis for the prime-vanishing subspace
# The prime evaluation matrix has one row per prime, one col per basis element.
# Null space = all coefficient vectors whose expression vanishes at every prime.
primes_small = filter(is_prime_trial, collect(2:50))
prime_mat = eval_matrix(2, 2, primes_small)   # (d=2, a_max=2)
null_vecs = rational_nullspace(prime_mat)
println("Null space dimension at (d=2, a_max=2): $(size(null_vecs, 2))")
println("(This is the prime-vanishing subspace dimension over ℚ.)")

In [ ]:
# test_conjecture: full pipeline for a given (d, a_max)
# Returns: holds (Bool), dim_basis, dim_prime_vanishing, dim_table1_span, counterexample

println("Conjecture test results:")
println("─" ^ 65)
@printf "  %8s  %8s  %6s  %8s  %8s  %s\n" "d" "a_max" "basis" "pv_dim" "t1_rank" "result"
println("  ", "─" ^ 62)

for (d, a) in [(2, 2), (3, 4), (4, 4)]
    r = test_conjecture(d, a; N=150, verbose=false)
    status = r.holds ? "holds ✓" : "COUNTEREXAMPLE"
    @printf "  %8d  %8d  %6d  %8d  %8d  %s\n" d a r.dim_basis r.dim_prime_vanishing r.dim_table1_span status
end

In [ ]:
# At a_max=5: M_5 generates new prime-vanishing directions requiring E5 (unknown)
println("With a_max=5 (M₅ included), E5 is required:")
r5 = test_conjecture(3, 5; N=150, verbose=false)
@printf "  (d=3, a_max=5): pv_dim=%d, t1_rank=%d  → E5 needed (unknown)\n" r5.dim_prime_vanishing r5.dim_table1_span

println()
println("Interpretation:")
println("  a_max ≤ 4: conjecture confirmed numerically using E1..E4")
println("  a_max = 5: new dimension found — E5 (the unknown 5th Table 1 entry) is needed")
println("  Finding E5 explicitly is an open research problem.")

In [ ]:
# scan_conjecture: systematic sweep over a grid of (d, a_max) pairs
# Stops at the first counterexample.
# For the a_max≤4 case this should report all holds.
println("Systematic sweep: scan_conjecture(4, 4)")
println("─" ^ 55)
scan_conjecture(4, 4; N=150)

## Summary

This notebook demonstrated all major components of `QuasiShuffleAlgebra.jl`:

| Component | Functions | Section |
|---|---|---|
| Types | `Word`, `ZqElem`, `zq_add`, `zq_scale`, `zq_sub` | 2 |
| Divisor sums | `σ(k, n)` | 3 |
| MacMahon functions | `M1`, `M2`, `M3`, `M_direct` | 3, 5 |
| MacMahonesque | `M_macmahonesque(vec_a, n, [T])` | 4 |
| Prime detection | `E1`–`E4`, `is_prime_partition`, `verify_range` | 6, 7 |
| Bernoulli numbers | `bernoulli(n)` | 8 |
| Diamond product | `diamond_coeffs`, `diamond` | 9 |
| Quasi-shuffle | `quasishuffle_words`, `zq_multiply` | 10, 11, 12 |
| D operator | `d_operator(vec_a)` | 13 |
| Symmetrisation | `symmetrise(vec_a)` | 14 |
| Word enumeration | `all_words_up_to_weight` | 15 |
| Conjecture test | `build_basis`, `eval_matrix`, `table1_coeffs`, `rational_rref!`, `rational_nullspace`, `rank_over_Q`, `is_in_colspan`, `test_conjecture`, `scan_conjecture` | 16 |

### Implementation notes

- **Weight convention**: `vec_a[1]` is the exponent of the **largest** part, matching all explicit formulas in the paper (opposite of the definition notation).
- **D operator solver**: uses RREF on augmented matrix `[A | b]` over `Rational{BigInt}`; returns a particular solution when the system is underdetermined.
- **Conjecture test**: uses evaluations of `E1`–`E4` directly (not truncated coefficient vectors), and checks membership in the $\mathbb{Q}[n]$-span via column-span rank test.
- **E5**: the fifth Table 1 entry is unknown; `test_conjecture` shows it is needed when `a_max = 5`.

### References

- Craig, van Ittersum & Ono, *"Integer Partitions Detect the Primes"*, arXiv:2405.06451v2, 2024
- Bachmann & Kühn, *"The algebra of multiple divisor functions"*, Ramanujan J. 40 (2016)
- MacMahon, *Combinatory Analysis*, 1916